# Introduction to Deep Learning, Assignment 2, Task 2

# Introduction

The goal of this assignment is to learn how to use encoder-decoder recurrent neural networks (RNNs) in **PyTorch**. Specifically we will be dealing with a sequence-to-sequence problem and try to build recurrent models that can learn the principles behind simple arithmetic operations (**integer addition and subtraction**).

<img src="https://i.ibb.co/5Ky5pbk/Screenshot-2023-11-10-at-07-51-21.png" alt="Screenshot-2023-11-10-at-07-51-21" border="0" width="500">

When using the Lab Computers, download the Jupyter Notebook to one of the machines first.

If you want to use Google Colab, you should first copy this notebook and enable GPU runtime in 'Runtime -> Change runtime type -> Hardware acceleration -> GPU'.


# Function definitions for creating the datasets

First we need to create our datasets that are going to be used for training our models.

In order to create image queries of simple arithmetic operations such as '15+13' or '42-10' we need to create images of '+' and '-' signs using the ***opencv*** library. We will use these operand signs together with the MNIST dataset to represent the digits.


In [31]:
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy.ndimage import rotate
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torchvision

# Set device (use GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

%matplotlib inline

Using device: cuda


In [2]:
# Create plus/minus operand signs
def generate_images(number_of_images=50, sign='-'):
    blank_images = np.zeros([number_of_images, 28, 28])  # Dimensionality matches the size of MNIST images (28x28)
    x = np.random.randint(12, 16, (number_of_images, 2)) # Randomized x coordinates
    y1 = np.random.randint(6, 10, number_of_images)       # Randomized y coordinates
    y2 = np.random.randint(18, 22, number_of_images)      # Randomized y coordinates

    for i in range(number_of_images): # Generate line coordinates
        cv2.line(blank_images[i], (y1[i], x[i,0]), (y2[i], x[i, 1]), (255,0,0), 2, cv2.LINE_AA)
        if sign == '+':
            cv2.line(blank_images[i], (x[i,0], y1[i]), (x[i, 1], y2[i]), (255,0,0), 2, cv2.LINE_AA)
        if sign == '*':
            cv2.line(blank_images[i], (x[i,0], y1[i]), (x[i, 1], y2[i]), (255,0,0), 2, cv2.LINE_AA)
            blank_images[i] = rotate(blank_images[i], -50, reshape=False)
            cv2.line(blank_images[i], (x[i,0], y1[i]), (x[i, 1], y2[i]), (255,0,0), 2, cv2.LINE_AA)
        if sign == '/':
            cv2.line(blank_images[i], (x[i,0], y1[i]), (x[i, 1], y2[i]), (255,0,0), 2, cv2.LINE_AA)
            blank_images[i] = rotate(blank_images[i], -50, reshape=False)

    return blank_images


In [3]:
def to_padded_chars(integer, max_len=3, pad_right=False):
    """
    Returns a string of len()=max_len, containing the integer padded with ' ' on either right or left side
    """
    length = len(str(integer))
    padding = (max_len - length) * ' '
    if pad_right:
        return str(integer) + padding
    else:
        return padding + str(integer)

def create_data(highest_integer, num_addends=2, operands=['+', '-']):
    """
    Creates the following data for all pairs of integers up to [1:highest integer][+/-][1:highest_integer]:

    @return:
    X_text: '51+21' -> text query of an arithmetic operation (5)
    X_img : Stack of MNIST images corresponding to the query (5 x 28 x 28) -> sequence of 5 images of size 28x28
    y_text: '72 ' -> answer of the arithmetic text query
    y_img : Stack of MNIST images corresponding to the answer (3 x 28 x 28) -> sequence of 3 images of size 28x28
    """
    num_indices = [np.where(MNIST_labels == i)[0] for i in range(10)]
    num_data = [MNIST_data[inds] for inds in num_indices]
    image_mapping = dict(zip(unique_characters[:10], num_data))
    image_mapping['-'] = generate_images()
    image_mapping['+'] = generate_images(sign='+')
    image_mapping['*'] = generate_images(sign='*')
    image_mapping[' '] = np.zeros([1, 28, 28])

    X_text = []
    X_img = []
    y_text = []
    y_img = []

    for i in range(highest_integer + 1):      # First addend
        for j in range(highest_integer + 1):  # Second addend
            for sign in operands: # Create all possible combinations of operands
                query_string = to_padded_chars(str(i) + sign + str(j), max_len=max_query_length, pad_right=True)
                query_image = []
                for n, char in enumerate(query_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    query_image.append(image_set[index].squeeze())

                result = eval(query_string)
                result_string = to_padded_chars(result, max_len=max_answer_length, pad_right=True)
                result_image = []
                for n, char in enumerate(result_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    result_image.append(image_set[index].squeeze())

                X_text.append(query_string)
                X_img.append(np.stack(query_image))
                y_text.append(result_string)
                y_img.append(np.stack(result_image))

    return np.stack(X_text), np.stack(X_img)/255., np.stack(y_text), np.stack(y_img)/255.


# Creating our data

The dataset consists of 20000 samples (additions and subtractions between all 2-digit integers) and they have two kinds of inputs and label modalities:

  **X_text**: strings containing queries of length 5: ['  1+1  ', '11-18', ...]

  **X_image**: a stack of images representing a single query, dimensions: [5, 28, 28]

  **y_text**: strings containing answers of length 3: ['  2', '156']

  **y_image**: a stack of images that represents the answer to a query, dimensions: [3, 28, 28]


In [13]:
# Illustrate the generated query/answer pairs

unique_characters = '0123456789+- '       # All unique characters used in the queries (13 in total: digits 0-9, operands +, -, and space ' ')
highest_integer = 99                      # Highest value of integers contained in the queries

max_int_length = len(str(highest_integer))# Maximum number of characters in an integer
max_query_length = max_int_length * 2 + 1 # Maximum length of the query string (e.g. '22+10')
max_answer_length = 3                     # Maximum length of the answer string (e.g. '-98')

# Load MNIST dataset via torchvision
mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True)
MNIST_data = mnist_train.data.numpy()
MNIST_labels = mnist_train.targets.numpy()

# Create the data
X_text, X_img, y_text, y_img = create_data(highest_integer)
print("Data shapes:")
print(f"X_text: {X_text.shape}, X_img: {X_img.shape}, y_text: {y_text.shape}, y_img: {y_img.shape}")

## Display sample query/answer pairs
def display_sample(n):
    labels = ['X_img:', 'y_img:']
    for i, data in enumerate([X_img, y_img]):
        plt.subplot(1, 2, i + 1)
        plt.axis('off')
        plt.title(labels[i])
        plt.imshow(np.hstack(data[n]), cmap='gray')
    print('='*50, f'\nQuery #{n}\n\nX_text: "{X_text[n]}" = y_text: "{y_text[n]}"')
    plt.show()

for _ in range(3):
    display_sample(np.random.randint(0, len(X_text)))


Data shapes:
X_text: (20000,), X_img: (20000, 5, 28, 28), y_text: (20000,), y_img: (20000, 3, 28, 28)
Query #50

X_text: "0+25 " = y_text: "25 "
Query #19171

X_text: "95-85" = y_text: "10 "
Query #4423

X_text: "22-11" = y_text: "11 "


## Helper functions

The functions below will help with input/output of the data (one-hot encoding and decoding).


In [5]:
# One-hot encoding/decoding the text queries/answers so that they can be processed using RNNs

def encode_labels(labels, max_len=3):
    n = len(labels)
    length = len(labels[0])
    char_map = dict(zip(unique_characters, range(len(unique_characters))))
    one_hot = np.zeros([n, length, len(unique_characters)], dtype=np.float32)
    for i, label in enumerate(labels):
        for j, char in enumerate(label):
            one_hot[i, j, char_map[char]] = 1.0
    return torch.tensor(one_hot, dtype=torch.float32)

def decode_labels(logits_or_onehot):
    if isinstance(logits_or_onehot, torch.Tensor):
        pred = torch.argmax(logits_or_onehot, dim=2).cpu().numpy()
    else:
        pred = np.argmax(logits_or_onehot, axis=2)
    predicted = [''.join([unique_characters[i] for i in seq]) for seq in pred]
    return predicted

X_text_onehot = encode_labels(X_text)
y_text_onehot = encode_labels(y_text)

print(f"X_text_onehot shape: {X_text_onehot.shape}, y_text_onehot shape: {y_text_onehot.shape}")


X_text_onehot shape: torch.Size([20000, 5, 13]), y_text_onehot shape: torch.Size([20000, 3, 13])


---
---

## I. Text-to-Text RNN Model

The following code showcases how Recurrent Neural Networks (RNNs) are built using **PyTorch**. The architecture consists of:

1. **Encoder LSTM:** Encodes the input sequence into a context vector (last hidden state).
2. **Decoder Context Repeat:** Repeats the context vector across the $T=3$ output time steps.
3. **Decoder LSTM:** Processes the repeated sequence into hidden states at each output time step.
4. **Linear Output Layer:** Projects the hidden state at each time step to class logits over the 13 unique characters.

<img src="https://i.ibb.co/NY7FFTc/Screenshot-2023-11-10-at-09-27-25.png" alt="Screenshot-2023-11-10-at-09-27-25" border="0" width="500">


In [6]:
class Text2TextRNN(nn.Module):
    """
    Sequence-to-sequence RNN model for arithmetic queries in PyTorch
    """
    def __init__(self, vocab_size=len(unique_characters), hidden_size=256, max_target_len=max_answer_length):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.max_target_len = max_target_len
        
        # Encoder: reads query sequence of length 5
        self.encoder = nn.LSTM(input_size=vocab_size, hidden_size=hidden_size, batch_first=True)
        
        # Decoder: receives context vector repeated for 3 output steps
        self.decoder = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, batch_first=True)
        
        # Output projection to character vocabulary logits
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        # x: (Batch, seq_len=5, vocab_size=13)
        _, (h_n, _) = self.encoder(x) # h_n: (1, Batch, hidden_size)
        
        # Format context vector as (Batch, 1, hidden_size) and repeat 3 times
        context = h_n.permute(1, 0, 2)
        decoder_input = context.repeat(1, self.max_target_len, 1) # (Batch, 3, hidden_size)
        
        # Decode
        decoder_out, _ = self.decoder(decoder_input) # (Batch, 3, hidden_size)
        logits = self.fc(decoder_out)                # (Batch, 3, vocab_size)
        return logits


In [10]:
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_text_onehot, y_text_onehot, test_size=0.2, random_state=42
)

# Convert one-hot target tensors to integer class indices for CrossEntropyLoss: (N, 3)
y_train_indices = torch.argmax(y_train, dim=2)
y_test_indices = torch.argmax(y_test, dim=2)

train_dataset = TensorDataset(X_train, y_train_indices)
test_dataset = TensorDataset(X_test, y_test_indices)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Instantiate model, optimizer, and loss criterion
model = Text2TextRNN(vocab_size=len(unique_characters), hidden_size=256).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Training loop
epochs = 70
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        logits = model(batch_x) # (Batch, 3, 13)
        # Flatten batch and sequence dimensions for cross entropy
        loss = criterion(logits.view(-1, len(unique_characters)), batch_y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_x)

    # Evaluate sequence accuracy on test set
    model.eval()
    correct_sequences = 0
    total_sequences = 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            preds = torch.argmax(logits, dim=2) # (Batch, 3)
            correct_sequences += (preds == batch_y).all(dim=1).sum().item()
            total_sequences += len(batch_x)

    seq_acc = correct_sequences / total_sequences
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {total_loss/len(train_dataset):.4f} - Test Seq Accuracy: {seq_acc*100:.2f}%")

# Inspect predictions on sample queries
model.eval()
with torch.no_grad():
    sample_inputs = X_test[:10].to(device)
    sample_logits = model(sample_inputs)
    decoded_inputs = decode_labels(sample_inputs)
    decoded_preds = decode_labels(sample_logits)
    decoded_trues = decode_labels(y_test[:10])

    print("\nSample Predictions:")
    for query, pred, true in zip(decoded_inputs, decoded_preds, decoded_trues):
        status = "✓" if pred == true else "✗"
        print(f"Query: '{query}' | Predicted: '{pred}' | True: '{true}' [{status}]")


Epoch 1/70 - Train Loss: 2.1108 - Test Seq Accuracy: 0.92%
Epoch 2/70 - Train Loss: 1.6850 - Test Seq Accuracy: 1.65%
Epoch 3/70 - Train Loss: 1.4824 - Test Seq Accuracy: 3.80%
Epoch 4/70 - Train Loss: 1.3675 - Test Seq Accuracy: 2.97%
Epoch 5/70 - Train Loss: 1.2813 - Test Seq Accuracy: 4.47%
Epoch 6/70 - Train Loss: 1.2222 - Test Seq Accuracy: 4.95%
Epoch 7/70 - Train Loss: 1.1772 - Test Seq Accuracy: 4.95%
Epoch 8/70 - Train Loss: 1.1475 - Test Seq Accuracy: 3.92%
Epoch 9/70 - Train Loss: 1.1224 - Test Seq Accuracy: 5.27%
Epoch 10/70 - Train Loss: 1.0831 - Test Seq Accuracy: 5.55%
Epoch 11/70 - Train Loss: 1.0675 - Test Seq Accuracy: 5.88%
Epoch 12/70 - Train Loss: 1.0362 - Test Seq Accuracy: 8.55%
Epoch 13/70 - Train Loss: 1.0018 - Test Seq Accuracy: 9.68%
Epoch 14/70 - Train Loss: 0.9951 - Test Seq Accuracy: 8.62%
Epoch 15/70 - Train Loss: 0.9630 - Test Seq Accuracy: 8.20%
Epoch 16/70 - Train Loss: 0.9620 - Test Seq Accuracy: 10.20%
Epoch 17/70 - Train Loss: 0.9152 - Test Seq Accu

---
---

## II. Image-to-Text RNN Model

In this task, your model receives a sequence of MNIST images ($5 \times 28 \times 28$) representing the arithmetic query and predicts the result as a text string (length 3).

**Hints:**
- Construct an encoder that processes each image in the sequence using a CNN feature extractor (or linear layer) followed by an `nn.LSTM`.
- The decoder can follow the same architecture as the Text-to-Text model to produce character logits.


In [55]:
## Your code: Implement and evaluate the Image-to-Text RNN model
class Image2TextRNN(nn.Module):
    """
    Image-to-sequence RNN model for arithmetic queries in PyTorch
    """
    def __init__(self, vocab_size=len(unique_characters), hidden_size=256, max_target_len=max_answer_length):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.max_target_len = max_target_len
        
        # CNN applied independently to each 28x28 image
        self.cnn = nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2),  # 28 -> 14
    
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),  # 14 -> 7
    
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2),  # 7 -> 3
    
        nn.Flatten(),
    )
    
        self.cnn_output_size = 128 * 3 * 3
        self.image_to_feature = nn.Linear(self.cnn_output_size, hidden_size)

        # Encoder LSTM: processes the sequence of 5 image embeddings
        self.encoder = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, batch_first=True)
        
        # Decoder: receives context vector repeated for 3 output steps
        self.decoder = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, batch_first=True)
        
        # Output projection to character vocabulary logits
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        # x: (B, 5, 1, 28, 28)
        batch_size, num_images, channels, height, width = x.shape

        # Combine batch and image-sequence dimensions for the CNN
        x = x.reshape(
            batch_size * num_images, channels, height, width
        )                                        # (B*5, 1, 28, 28)

        cnn_features = self.cnn(x)               # (B*5, 3136)
        image_embeddings = self.image_to_feature(cnn_features)

        # Restore sequence dimension: (B, 5, hidden_size)
        image_embeddings = image_embeddings.reshape(
            batch_size, num_images, self.hidden_size
        )

        # Encode the 5-image sequence
        _, (h_n, _) = self.encoder(image_embeddings)  # h_n: (1, B, hidden_size)

        # Same decoder input construction as before
        context = h_n.permute(1, 0, 2)           # (B, 1, hidden_size)
        decoder_input = context.repeat(
            1, self.max_target_len, 1
        )                                        # (B, max_target_len, hidden_size)

        decoder_out, _ = self.decoder(decoder_input)
        logits = self.fc(decoder_out)            # (B, max_target_len, vocab_size)

        return logits

In [61]:
X_images = torch.tensor(X_img, dtype=torch.float32)

# Add channel dimension: (N, 5, 1, 28, 28)
X_images = X_images.unsqueeze(2)

X_img_train, X_img_test, X_text_train, X_text_test, y_train, y_test = train_test_split(
    X_images,
    X_text_onehot,
    y_text_onehot,
    test_size=0.2,
    random_state=42
)

y_train_indices = torch.argmax(y_train, dim=2).long()
y_test_indices = torch.argmax(y_test, dim=2).long()

train_dataset = TensorDataset(X_train, y_train_indices)
test_dataset = TensorDataset(X_test, y_test_indices)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Instantiate model, optimizer, and loss criterion
model = Image2TextRNN(vocab_size=len(unique_characters), hidden_size=256).to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

# Training loop
epochs = 70
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        logits = model(batch_x) # (Batch, 3, 13)
        # Flatten batch and sequence dimensions for cross entropy
        loss = criterion(logits.reshape(-1, len(unique_characters)), batch_y.reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_x)

    # Evaluate sequence accuracy on test set
    model.eval()
    correct_sequences = 0
    total_sequences = 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            preds = torch.argmax(logits, dim=2) # (Batch, 3)
            correct_sequences += (preds == batch_y).all(dim=1).sum().item()
            total_sequences += len(batch_x)

    seq_acc = correct_sequences / total_sequences
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {total_loss/len(train_dataset):.4f} - Test Seq Accuracy: {seq_acc*100:.2f}%")

# Inspect predictions on sample queries
model.eval()
with torch.no_grad():
    sample_images = X_img_test[:10].to(device)
    sample_queries = X_text_test[:10]
    sample_logits = model(sample_images)

    decoded_queries = decode_labels(sample_queries)
    decoded_preds = decode_labels(sample_logits)
    decoded_trues = decode_labels(y_test[:10])
    
    print("\nSample Predictions:")
    for i, (pred, true) in enumerate(zip(decoded_preds, decoded_trues)):
        status = "✓" if pred == true else "✗"
        print(f"Sample {i} | Predicted: '{pred}' | True: '{true}' [{status}]")

Epoch 1/70 - Train Loss: 2.1197 - Test Seq Accuracy: 0.50%
Epoch 2/70 - Train Loss: 1.9175 - Test Seq Accuracy: 0.68%
Epoch 3/70 - Train Loss: 1.7583 - Test Seq Accuracy: 0.73%
Epoch 4/70 - Train Loss: 1.6418 - Test Seq Accuracy: 1.27%
Epoch 5/70 - Train Loss: 1.5574 - Test Seq Accuracy: 2.00%
Epoch 6/70 - Train Loss: 1.4860 - Test Seq Accuracy: 2.05%
Epoch 7/70 - Train Loss: 1.4182 - Test Seq Accuracy: 2.70%
Epoch 8/70 - Train Loss: 1.3599 - Test Seq Accuracy: 4.03%
Epoch 9/70 - Train Loss: 1.3053 - Test Seq Accuracy: 3.40%
Epoch 10/70 - Train Loss: 1.2707 - Test Seq Accuracy: 3.72%
Epoch 11/70 - Train Loss: 1.2282 - Test Seq Accuracy: 4.38%
Epoch 12/70 - Train Loss: 1.1876 - Test Seq Accuracy: 4.62%
Epoch 13/70 - Train Loss: 1.1549 - Test Seq Accuracy: 4.92%
Epoch 14/70 - Train Loss: 1.1258 - Test Seq Accuracy: 4.85%
Epoch 15/70 - Train Loss: 1.0988 - Test Seq Accuracy: 5.20%
Epoch 16/70 - Train Loss: 1.0868 - Test Seq Accuracy: 6.50%
Epoch 17/70 - Train Loss: 1.0554 - Test Seq Accur

---
---

## III. Text-to-Image RNN Model

In this task, your model receives a text query string ($5 \times 13$) as input and generates a sequence of 3 images ($3 \times 28 \times 28$) corresponding to the answer digits.

**Hints:**
- Use a text LSTM encoder to produce a context vector.
- The decoder can use `nn.LSTM` followed by deconvolutional layers (`nn.ConvTranspose2d`) to upscale feature maps into $28 \times 28$ images for each time step.


In [ ]:
## Your code: Implement and evaluate the Text-to-Image RNN model
